# AAPL — 1-minute bars → bronze

Extraction of every field Alpaca exposes for AAPL 1-minute bars, from 2016 to today,
into `lakehouse.bronze` of the Unity Catalog lakehouse.

Fixed decisions (§1 of the design document):

| Decision | Value | Why |
|---|---|---|
| Source | Alpaca Market Data v2, `feed=sip` | 100% of the volume; available on the Basic plan for data older than 15 minutes. |
| Granularity | `1Min` | ~1M rows for AAPL 2016→2026, re-aggregable to 5m/15m/1h/1d without touching the API again. |
| Adjustment | `raw` | Alpaca adjusts with the factors known *today*; storing raw keeps the dataset reproducible. Adjustment happens in silver. |
| Format | Delta Lake, external table in Unity Catalog | Atomic per-partition commits make a retry safe; the catalog makes the same table readable from pandas, Spark and the UC UI. |
| Timezone | UTC | `America/New_York` only appears in silver, where the market calendar makes local time meaningful. |

The layer boundary matters: a bug in a feature definition must never force a
re-download of ten years of history through a 200 req/min budget.

## Utils

### Libraries & paths

In [ ]:
import logging
import os
import pathlib
import sys

from dotenv import load_dotenv

# The notebook lives under scripts/dwh/<layer>/<source>/; walk up to the data-etl root.
DATA_ETL_ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "src" / "extractions").is_dir())
sys.path.insert(0, str(DATA_ETL_ROOT))

load_dotenv(DATA_ETL_ROOT.parent / ".env")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s")

from src.dwh.bronze import alpaca_bars, alpaca_reference
from src.extractions.alpaca import MarketData
from src.storage.catalog import get_catalog

### Parameters

In [ ]:
from datetime import date

SYMBOLS = "AAPL"          # the schema is multi-symbol from day one: "AAPL,SPY,QQQ" also works
TIMEFRAME = "1Min"
FEED = "sip"
ADJUSTMENT = "raw"
START_DATE = date(2016, 1, 1)
END_DATE = date.today()

CREDENTIALS = {
    "APCA-API-KEY-ID": os.getenv("APCA-API-KEY-ID"),
    "APCA-API-SECRET-KEY": os.getenv("APCA-API-SECRET-KEY"),
}

# UNITY_CATALOG_URI / UNITY_CATALOG_NAME / LAKEHOUSE_ROOT come from the environment.
lakehouse = get_catalog()
market = MarketData(credentials=CREDENTIALS)

print(f"{SYMBOLS} {TIMEFRAME} {START_DATE} \u2192 {END_DATE} (feed={FEED}, adjustment={ADJUSTMENT})")
print(f"catalog {lakehouse.catalog} at {lakehouse.client.uri}, tables under {lakehouse.warehouse_root}")

## Support tables

### `dim_market_calendar`

Without it there is no way to tell "a minute with no trades" from "the market was
closed", and that distinction decides how gaps are filled in silver. Early closes at
13:00 ET (Thanksgiving eve, 24 Dec) are real and frequent.

In [ ]:
calendar = alpaca_reference.ingest_market_calendar(
    market, start=START_DATE, end=END_DATE, lakehouse=lakehouse
)

print(f"{len(calendar)} sessions, {int(calendar['is_half_day'].sum())} of them early closes")
calendar.head()

### `dim_corporate_actions`

AAPL in this range: the 4:1 split of 2020-08-31 and quarterly dividends. Both create
discontinuities that would otherwise be learned as signal.

In [ ]:
corporate_actions = alpaca_reference.ingest_corporate_actions(
    market, symbols=SYMBOLS, start=START_DATE, end=END_DATE, lakehouse=lakehouse
)

corporate_actions.groupby("type").size()

In [ ]:
corporate_actions[corporate_actions["type"] == "split"]

## Bronze — `fact_bars_raw`

One request per month (`limit=10000`, `next_page_token`), each write aligned with exactly one
`symbol=/year=/month=` partition. Partitions are rewritten whole rather than appended to,
so a retry after a partial failure cannot duplicate rows.

Set `is_overwrite=False` to resume an interrupted backfill without re-hitting the API.

In [ ]:
summary = alpaca_bars.ingest_bars_raw(
    market,
    symbols=SYMBOLS,
    start=START_DATE,
    end=END_DATE,
    timeframe=TIMEFRAME,
    feed=FEED,
    adjustment=ADJUSTMENT,
    lakehouse=lakehouse,
    is_overwrite=True,
)

print(f"{summary['row_count'].sum():,} bars written across {len(summary)} monthly partitions")
summary.tail()

## Verification

In [ ]:
bars = alpaca_bars.read_bars_raw(symbol="AAPL", lakehouse=lakehouse)

print(f"{len(bars):,} rows, {bars['timestamp_at'].min()} \u2192 {bars['timestamp_at'].max()}")
bars.groupby("year").size()

In [ ]:
# Every field the API returns, plus the request provenance that makes an audit possible.
bars.head()

In [ ]:
# The logical key (symbol, timestamp_at, feed) must be unique for the ingestion to be idempotent.
duplicate_count = bars.duplicated(subset=["symbol", "timestamp_at", "feed"]).sum()
print(f"duplicates on (symbol, timestamp_at, feed): {duplicate_count}")

## Catalog

The three tables are external Delta tables registered in Unity Catalog, so Spark
reads them as `lakehouse.bronze.<table>` without knowing any path.


In [ ]:
for table in lakehouse.client.list_tables(lakehouse.catalog, "bronze"):
    print(f"{lakehouse.catalog}.bronze.{table['name']:24} {table['storage_location']}")
